## AlphaEarth Foundations

You can read more about AlphaEarth Here: 
https://deepmind.google/blog/alphaearth-foundations-helps-map-our-planet-in-unprecedented-detail/

or 

Here: https://developers.google.com/earth-engine/tutorials/community/satellite-embedding-01-introduction

AlphaEarth Foundation Model represents a paradigm shift in how we analyze the Earth's surface. Developed by Google DeepMind, AlphaEarth provides what are called "embeddings", simialr to spectral bands, provide a semantic "fingerprint" of every 10-meter patch of the planet.




### What are embeddings

Embeddings are a way to compress large amounts of information into a smaller set of features that represent meaningful semantics. The AlphaEarth Foundations model takes time series of images from sensors including Sentinel-2, Sentinel-1, and Landsat and learns how to uniquely represent the mutual information between sources and targets with just 64 numbers (learn more in the paper). The input data stream contains thousands of image bands from multiple sensors and the model takes this high dimensional input and turns it into a lower dimensional representation.

A good mental model to understand how AlphaEarth Foundations works is a technique called Principal Component Analysis (PCA). PCA also helps reduce the dimensionality of the data for machine learning applications. While PCA is a statistical technique and can compress tens of input bands into a handful of principal components, AlphaEarth Foundations is a deep-learning model that can take thousands of input dimensions of multi-sensor time-series datasets and learns to create a 64-band representation that uniquely captures the spatial and temporal variability of that pixel.

An embedding field is the continuous array or “field” of learned embeddings. Images in the embedding fields collections represent space-time trajectories covering an entire year and have 64 bands (one for each embedding dimension).

###  Technical Specifications
- Spatial Resolution: 10 meters per pixel (aligned with the Sentinel-2 grid).
- Dimensions: 64 numerical values per pixel (Bands A00 through A63).
- Temporal Coverage: Annual mosaics from 2017 to the 2025.
- Unit-Length Vectors: All vectors are normalized, meaning you can calculate similarity using simple dot products.

### Google Earth Engine

Google Earth Engine (GEE) is a cloud-based platform designed for processing and analyzing geospatial data on a large scale.

It provides access to a vast collection of public datasets, including satellite imagery, aerial photography, weather data, and other geospatial information. GEE allows users to run computations on these datasets using its powerful cloud infrastructure, making it ideal for environmental monitoring, research, and analysis.

AlphaEarth's embeddings are hosted at Google Earth Engine:

https://developers.google.com/earth-engine/datasets/catalog/GOOGLE_SATELLITE_EMBEDDING_V1_ANNUAL?utm_source=deepmind.google&utm_medium=referral&utm_campaign=gdm&utm_content#description

### Get Started

In this tutorial, I will demostrate how to use the embbedings from AlphaEarth for two tasks:

1. Using similarity to identify lakes in Tallahassee Area.

2. Land cover change between 2017 and 2025.

Install the libraries if haven't

In [1]:
#pip install earthengine-api geemap

The below code will ask you to log in to google and select your google earth engine project. It will generate a long text API code then you need to copy and paste it to the box below.

The authenticatation only needs to be ran once.

In [2]:
import ee
import geemap
import numpy as np
import matplotlib.pyplot as plt

# 1. Authenticate and Initialize
# Trigger the authentication flow.
ee.Authenticate()

# Initialize the library.
ee.Initialize()


### Define the Area of Interest (Tallahassee, FL)

In [3]:
# Coordinates for downtown Tallahassee
lon, lat = -84.2807, 30.4383
tallahassee_point = ee.Geometry.Point([lon, lat])
# Buffer to create a region (e.g., 10km radius)
tallahassee_area = tallahassee_point.buffer(10000).bounds()


### Load AlphaEarth Foundations Embeddings

In [4]:
# The dataset ID is 'GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL'
dataset = ee.ImageCollection("GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL")

# Filter for the desired year (2025) and location

# Each image represents a full calendar year (2017-2025)
embeddings_2026 = dataset.filterDate('2025-01-01', '2026-01-01') \
                         .filterBounds(tallahassee_area) \
                         .mosaic() \
                         .clip(tallahassee_area)

### Visualizing the Embeddings

You will see different landcovers have different embedding signatures.

In [5]:
# Since there are 64 bands (A00 to A63), we map 3 bands to RGB 
# to see a "fingerprint" of the land cover.
vis_params = {
    'min': -0.3, 
    'max': 0.3, 
    'bands': ['A01', 'A16', 'A09'] # Common "colorful" band combo
}

# 6. Display using geemap
Map = geemap.Map()
Map.centerObject(tallahassee_point, 12)
Map.addLayer(embeddings_2026, vis_params, 'AlphaEarth 2025 Embeddings')
Map

Map(center=[30.438299999999998, -84.2807], controls=(WidgetControl(options=['position', 'transparent_bg'], pos…

### Semantic Similarity Search ("Find more of this")


Embeddings present a unique opportunity to find similar locations and features using Earth observation data. By comparing the embedding vector of a reference location with the embedding vectors for all other pixels of an embedding image, we can find locations that exhibit similar properties as the reference location. In practice, this allows us to easily find objects or particular types of sites in our region of interest.


As an example, I selected the lake in the FSU Lakefront park. Its coordinates are [-84.33999808470628, 30.405755561041907].


In [6]:
# Reference Point: FSU Lakefront park
lake_ref = ee.Geometry.Point([-84.33999808470628, 30.405755561041907])

# Extract the 64D embeddings for the lake around a 50m buffer.
# We average the pixels in the immediate area to get a robust signature
lake_sample = embeddings_2026.select('A.*').reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=lake_ref.buffer(50),
    scale=10
)
lake_vector = ee.Array(ee.Dictionary(lake_sample).values())

# 5. Perform Semantic Search
# Calculates cosine similarity across the entire Tallahassee area
similarity = embeddings_2026.select('A.*').toArray().arrayDotProduct(lake_vector)


### Visuzliation

We can now visualize the pixels that are similar to the lake we used. We can set a threshold as 0.95, which is 95% similar. You can try other thresholds but will be more revealing.

By running the code below, you will see the water bodies are highlighted as yellow. You may also observse some other water bodies are not highlighted, I think because the lake we selected is a clear lake, some other lakes for example northern part of the Lake Jackson that has high diversity of aquatic vegetation (more grassy) is not showing up.

In [7]:
threshold_value = 0.95

# Create a binary mask:
# Pixels >= threshold become 1 (keep), others become 0 (hide)
mask = similarity.gte(threshold_value)

# Update the similarity image mask. 
# Earth Engine makes pixels with a mask of 0 transparent.
masked_similarity = similarity.updateMask(mask)
# ==============================================================================

# 6. Display Results
Map = geemap.Map()
Map.setCenter(-84.3001, 30.4439, 12) # Zoomed in on FSU 

# Add satellite hybrid basemap for context
Map.add_basemap('SATELLITE')

# Visualization Parameters
#Mask out all other pixels with similarity < the threshold, and show similar water pixels as yellow
vis_masked = {
    'min': threshold_value, 
    'max': 1.0, 
    'palette': ['yellow']
}

Map.addLayer(masked_similarity, vis_masked, 'Lakes (Thresholded >0.95)')
Map

Map(center=[30.4439, -84.3001], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topr…

### Land cover change between 2017 and 2025.

In [8]:
# Load AlphaEarth Embeddings & Calculate Change
ae_dataset = ee.ImageCollection("GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL")
ae_2017 = ae_dataset.filterDate('2017-01-01', '2017-12-01').mosaic().clip(tallahassee_area)
ae_2025 = ae_dataset.filterDate('2025-01-01', '2025-12-01').mosaic().clip(tallahassee_area)

# Similarity calculation
similarity = ae_2017.multiply(ae_2025).reduce(ee.Reducer.sum())


In [9]:
#Apply a mask to only display similarity < the threshold. Things remain stable are masked out.
mask = similarity.lt(0.80) 
detected_change = similarity.updateMask(mask)

In [10]:
# Load Sentinel-2 Satellite Imagery for before and after as basemaps
def get_sat_mosaic(year):
    return ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
        .filterBounds(tallahassee_area) \
        .filterDate(f'{year}-01-01', f'{year}-03-01') \
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10)) \
        .median() \
        .clip(tallahassee_area)

sat_2017 = get_sat_mosaic(2017)

sat_2025 = get_sat_mosaic(2025)


In [11]:
# Define Visualization Parameters
sat_vis = {'min': 0, 'max': 1000, 'bands': ['B4', 'B3', 'B2']}
change_vis = {'min': 0.5, 'max': 0.9, 'palette': ['red', 'orange', 'yellow']}


In [12]:
# PREPARE THE VISUALIZED LAYERS
# Left Side: Pure 2017 Satellite
left_viz = sat_2017.visualize(**sat_vis)

In [13]:
# Right Side: 2025 Satellite with the Mask "Pinned" on top
# We visualize both first, then blend. The mask sits on top of the satellite.
right_viz = sat_2025.visualize(**sat_vis).blend(detected_change.visualize(**change_vis,opacity=0.9))

# Render Swipe Map
Map = geemap.Map()
Map.centerObject(tallahassee_area, 12)

# Convert visualized images to TileLayers
left_layer = geemap.ee_tile_layer(left_viz, {}, 'Satellite 2017')
right_layer = geemap.ee_tile_layer(right_viz, {}, 'Satellite 2025 + Mask')

# Initialize split-screen view
Map.split_map(left_layer, right_layer)

Map.add_text("Left: 2017 Raw | Right: 2025 + AlphaEarth Change Mask", position='bottomright')
Map

Map(center=[30.43830326440559, -84.2805248237281], controls=(ZoomControl(options=['position', 'zoom_in_text', …

Areas with higher changes in the embedding space are highlighted in oragnge to red.